<a href="https://colab.research.google.com/github/AngelTroncoso/Feature-Engineering/blob/main/actividad5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformaciones de Potencia y Logarítmicas

¡Hola! Qué bueno que estés aquí para aprender una de las técnicas más potentes en la ingeniería de características. Hoy vamos a domar esos datos rebeldes que no se portan como los modelos quieren, permitiendo que tus predicciones pasen de ser mediocres a ser brillantes gracias a la normalización matemática.

### ¿Por qué necesitamos normalizar?
Muchos de nuestros modelos favoritos, como la Regresión Lineal, son como personas muy estructuradas que necesitan que la información sea simétrica. Cuando tenemos datos con mucho sesgo, es decir, acumulados en un extremo, el modelo se confunde. Es como intentar leer un libro donde todas las letras están amontonadas en una esquina de la página; simplemente no puedes extraer la esencia de la historia de forma correcta si no distribuyes el contenido.

**Objetivo de la sesión:** Aplicar transformaciones logarítmicas y de Box-Cox en Python para normalizar distribuciones sesgadas, facilitando que los datos cumplan con los supuestos de linealidad requeridos por muchos modelos.

### 1. Preparación del Escenario: El Problema del Sesgo
Imagina un conjunto de datos de precios de venta de casas. Tenemos valores que van desde los cincuenta mil hasta los dos millones de dólares. La gran mayoría de las casas están concentradas en el rango bajo, creando una *montaña* a la izquierda y una cola larguísima a la derecha (sesgo positivo). Este sesgo hace que nuestro promedio sea engañoso.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Simulamos datos de precios de casas con un fuerte sesgo positivo
np.random.seed(42)
datos_precios = np.random.exponential(scale=150000, size=1000) + 50000

df = pd.DataFrame({'precio': datos_precios})

print('Primeros registros de precios:')
print(df.head())
print()

# Visualizamos la distribución original
plt.figure(figsize=(10, 5))
plt.hist(df['precio'], bins=50, color='skyblue', edgecolor='black')
plt.title('Distribución Original de Precios (Sesgo Positivo)')
plt.xlabel('Precio en Dólares')
plt.ylabel('Frecuencia')
plt.show()

### 2. La Transformación Logarítmica
Su magia reside en cómo comprime las escalas. Imagina que el logaritmo actúa como un acordeón: estira los valores pequeños que están apretados y encoge los valores muy grandes que están lejos. Al aplicar esta función, los números que crecen de forma exponencial se vuelven lineales.

Utilizaremos la función `log1p` de Numpy, que aplica $log(1+x)$ para evitar errores si existen valores en cero.

In [ ]:
# Aplicamos la transformación logarítmica
df['precio_log'] = np.log1p(df['precio'])

print('Datos después de transformación logarítmica:')
print(df[['precio', 'precio_log']].head())
print()

# Visualizamos el resultado
plt.figure(figsize=(10, 5))
plt.hist(df['precio_log'], bins=50, color='salmon', edgecolor='black')
plt.title('Distribución después de Transformación Logarítmica')
plt.xlabel('Log(Precio)')
plt.ylabel('Frecuencia')
plt.show()

### 3. Transformación Box-Cox: El Sastre Matemático
A veces, el logaritmo no es suficiente. Box-Cox es como un sastre que ajusta el traje a medida del dato. Utiliza un parámetro llamado **lambda** ($λ$) para probar diferentes potencias hasta encontrar la que mejor normaliza los datos.

**Caso de Estudio (Escenario):** Roberto, un científico de datos en una empresa de energía, usa Box-Cox para estabilizar la varianza de los registros de consumo eléctrico durante picos de demanda. Esto evitó errores enormes en días de calor extremo.

In [ ]:
# Aplicamos Box-Cox y obtenemos el valor óptimo de lambda
precios_boxcox, lmbda = stats.boxcox(df['precio'])
df['precio_boxcox'] = precios_boxcox

print('El valor de lambda optimizado es:')
print(lmbda)
print()

# Visualizamos con un gráfico Q-Q para ver la normalidad
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
stats.probplot(df['precio'], dist='norm', plot=plt)
plt.title('Gráfico Q-Q Original')

plt.subplot(1, 2, 2)
stats.probplot(df['precio_boxcox'], dist='norm', plot=plt)
plt.title('Gráfico Q-Q Post Box-Cox')

plt.tight_layout()
plt.show()

### 4. Manejo de Errores Comunes: El problema del Cero
Un error frecuente es intentar aplicar estas transformaciones a datos que contienen ceros o valores negativos. El logaritmo de cero no existe, y Box-Cox requiere estrictamente valores positivos.

In [ ]:
# Creamos una serie con un cero
datos_con_cero = pd.Series([10, 20, 0, 40, 50])

print('Intentando logaritmo estándar en datos con cero:')
with np.errstate(divide='ignore'):
    resultado_error = np.log(datos_con_cero)
    print(resultado_error)
print()

print('Solución usando log1p (suma 1 automáticamente):')
resultado_fijo = np.log1p(datos_con_cero)
print(resultado_fijo)

### Resumen y Conclusión
¿Te das cuenta de la diferencia fundamental entre ambos métodos? Mientras que el logaritmo es una transformación fija, Box-Cox es un proceso de optimización que busca la mejor forma posible de forma automática.

Sin embargo, recuerda la regla de oro: la interpretación. Al transformar, ya no estamos hablando de dólares originales. Si quieres explicar resultados a un cliente, deberás aplicar la operación inversa para volver a la escala original.

¡Muy bien! Has aprendido a transformar datos de forma profesional. Recuerda que la ingeniería de características es tanto un arte como una ciencia, y hoy has sumado dos herramientas fundamentales a tu cinturón de habilidades técnicas. ¡Nos vemos en la siguiente lección!